In [1]:
import numpy as np
from numpy.linalg import norm
import math, random
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

In [2]:
#spatial funcs for WoS

def closestPoint(v, s):
    u = s[1] - s[0]
    rate = max(0, min(u.dot(v - s[0])/u.dot(u), 1))
    return (s[0] + rate*(s[1] - s[0]))

def shortestDistance(v, segments):
    r = float("inf")
    for s in segments:
        u = closestPoint(v, s)
        if norm(v - u) < r:
            r = norm(v - u)
    return r

In [3]:
'''record info for all the valid paths
    a list of single path (start -> boundary end) (dynamic length), 
    a single boundary result, 
'''
def pathGenerator(v, segments, g, eps = 0.01, maxSteps = 16, maxR = 10):
    x0 = v
    path = []
    for step in range(0, maxSteps): 
        path.append(x0)
        r = min([shortestDistance(x0, segments), maxR])
        if r < eps: 
            return path, g(x0)
        theta = random.uniform(0, 2*math.pi)
        x0 = x0 + np.array([r * math.cos(theta), r * math.sin(theta)])
    return [], 0


In [11]:
target = []
width = range(1, 200)
height = range(1, 200)
for i in width:
    for j in height:
        target.append(np.array([i/100 - 1, j/100 - 1]))

print(len(target))

39601


In [12]:
# set up the problem
segments = [np.array([[-1, -1], [-1, 1]]), np.array([[-1, -1], [1, -1]]), np.array([[1, -1], [1, 1]]), np.array([[-1, 1], [1, 1]])]

#v = np.array([0, 0])
def boundary(v):
    return v[1] * v[0]

In [13]:
paths = []
boundaries = []
for v in target:
    p, b = pathGenerator(v, segments, boundary)
    if len(p) != 0:
        paths.append(p)
        boundaries.append(b)

In [17]:
len(paths)

37324

In [18]:
class ZNN(keras.Model):
    def __init__(self, units):
        super(ZNN, self).__init__()
        self.layer1 = keras.layers.Dense(units[0])
        self.layer2 = keras.layers.Dense(units[1])
        self.layer3 = keras.layers.Dense(units[2])
        self.outputs = layers.Dense(2)

    def call(self, input_tensor, training):
        x = self.layer1(input_tensor)
        x = tf.nn.relu(x)
        x = self.layer2(x)
        x = tf.nn.relu(x)
        x = self.layer3(x)
        x = tf.nn.relu(x)
        x = self.outputs(x)
        return x

In [59]:
class Ysolver(keras.Model):
    # set y as a trainable variable, initialized randomly
    def __init__(self, layerP):
        super(Ysolver, self).__init__()
        self.model = ZNN(layerP)
        self.y = tf.Variable(np.random.uniform(low=-1, high=1, size=[1, 1]),trainable = True, dtype ="float32")

    def call(self, path, training):
        y_pred = self.y
        for i in range(len(path) - 1): ###input path should be (-1, 2) shaped nparray, no len()
            z = self.model(np.array(path[i]).reshape(1,2).astype("float32"), training) #obtain gradients for steps
            y_pred.assign_add(tf.reduce_sum(z * (path[i + 1] - path[i]), 1, keepdims=True)) #accumulating
            print(y_pred)
        return y_pred

In [60]:
class NNsolver(keras.Model):
    def __init__(self, paths, boundary):
        super(NNsolver, self).__init__()
        self.paths = paths
        self.boundary = boundary
        self.model = Ysolver([32, 64, 128])
        self.optimizer = tf.keras.optimizers.legacy.Adam(learning_rate=1e-4 , epsilon=1e-8)

    def loss_fn(self, y_pred, y):
        delta = y_pred - y
        loss = tf.square(delta)
        return loss

    def train_step(self, x, training):
        return self.model(x, training)

    def train(self): 
        training_history = []
        for i in range(len(self.paths)):
            with tf.GradientTape() as tape:
                y_pred = self.train_step(np.array(self.paths[i]).reshape(-1, 2), training = True)
                loss = self.loss_fn(y_pred,self.boundary[i])

            training_vars = self.model.trainable_variables
            #print(loss)
            grad = tape.gradient(loss, training_vars)
            self.optimizer.apply_gradients(zip(grad, training_vars))
            
            training_history.append([i, loss])

        return training_history

    def test1(self, v):
        return self.model(v.reshape(1,2).astype("float32"), training = True)

    def test2(self, path):
        return self.train_step(path, training = True)

    def accuracy1(self, i):
        return abs((self.test2(self.paths[i]) - self.boundary[i])/self.boundary[i])

    def accuracy2(self):
        sum = 0
        for i in range(100):
            sum += self.accuracy1(i)
        return sum/100

In [61]:
training = NNsolver(paths, boundaries)
training.train()

<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.42053333]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.41773924]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.41492915]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.41515017]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.4150051]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.40973523]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.40546453]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.39669657]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.38940826]], dtype=float32)>
<tf.Variable 'Variable:0' shape=(1, 1) dtype=float32, numpy=array([[-0.38896763]], dtype=float32)>


[[0,
  <tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[1.9702816]], dtype=float32)>],
 [1,
  <tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[1.9229568]], dtype=float32)>],
 [2,
  <tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[1.9297918]], dtype=float32)>],
 [3,
  <tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[1.6700913]], dtype=float32)>],
 [4,
  <tf.Tensor: shape=(1, 1), dtype=float32, numpy=array([[1.8042759]], dtype=float32)>]]